In [ ]:
import datetime

def now():
    return datetime.datetime.now().strftime("%H:%M:%S")


In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

async def simple_coroutine(sleep_time: int=1):
    """A simple coroutine that yields control back to the event loop."""
    thread_name = asyncio.current_task().get_name()
    thread_id = id(asyncio.current_task())
    
    print(f"Starting {thread_name} (ID: {thread_id}): {now()}")
    await asyncio.sleep(sleep_time)  # Simulate an I/O operation
    print(f"Ending {thread_name} (ID: {thread_id}): {now()}")

    return (f"Thread {thread_id} slept for {sleep_time} seconds")


# Below coroutine invocation uses same thread id and name due to sequential await calls sharing the same task context
print ("\n==== Sequential Task Creation (same thread id & name)====\n")
await simple_coroutine(2)
await simple_coroutine(3)


# asyncio.run creates new tasks with its own context
print("\n==== Sequential Task creation (different thread id & name)====\n")
asyncio.run(simple_coroutine(2))
asyncio.run(simple_coroutine(3))


In [ ]:
# Execute the coroutines as separate tasks to ensure unique thread IDs and names
async def run_separate_tasks():
    # Create separate tasks for each coroutine call
    task1 = asyncio.create_task(simple_coroutine(2))
    task2 = asyncio.create_task(simple_coroutine(3))
    
    # Wait for both tasks to complete
    await task1
    await task2


# Below invocation creates separate tasks with unique thread id and name
print("\n====Separate Task run (new Thread id & name)====\n")
await run_separate_tasks()


In [ ]:
future_tuple= asyncio.gather(
    simple_coroutine(2),
    simple_coroutine(3)
)

print(await future_tuple)

In [ ]:
## Task group

async with asyncio.TaskGroup() as tg:
    task1 = tg.create_task(simple_coroutine(2))
    task2 = tg.create_task(simple_coroutine(3))

taskResult = asyncio.gather(task1, task2)

print(taskResult.result)
print(f"Task results are: \n", {result for result in taskResult.result()})
